# **Activity 2: Building the Data Processing Pipeline - Group 3**

**Group members:**
- Abdullah Ali Saleem
- Danil Seksenov
- Nikita Soo
- Fatih Özdil

##**Setup: Environment Preparation**

In [ ]:
# Install the necessary libraries
!pip install datasets pandas -q

**Group Discussion 1**

---

**Question:** In the code above, we used an exclamation mark (`!`) before `pip install`. In Colab, you might also occasionally see commands starting with a percent sign (`%` or `%%`). What is the difference between `!` and `%`, and why do we use them?

### ! Exclamation mark is for shell no the python enviroments
### % %% these marks Jupyter/Colab notebooks to control how the notebook behaves

##**Part 1: Inspecting and Deduplicating Data**

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load the dataset
dataset = load_dataset('fka/awesome-chatgpt-prompts', split='train')

# Convert to Pandas DataFrame
df = dataset.to_pandas()

print(f"Total number of rows before removing duplicates: {len(df)}")

# Remove duplicate rows based on the 'prompt' column
df_cleaned = df.drop_duplicates(subset=['prompt'])

print(f"Total number of rows after removing duplicates: {len(df_cleaned)}")

# Display the first few rows of the cleaned DataFrame
display(df_cleaned.head())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

prompts.csv: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Total number of rows before removing duplicates: 1559
Total number of rows after removing duplicates: 1557


,act,prompt,for_devs,type,contributor
0,Ethereum Developer,Imagine you are an experienced Ethereum develo...,True,TEXT,ameya-2003
1,Linux Terminal,I want you to act as a linux terminal. I will ...,True,TEXT,f
2,English Translator and Improver,"I want you to act as an English translator, sp...",False,TEXT,f
3,Job Interviewer,I want you to act as an interviewer. I will be...,False,TEXT,"f,iltekin"
4,JavaScript Console,I want you to act as a javascript console. I w...,True,TEXT,omerimzali


**Group Discussion 2**

---

**Question:** The code above successfully removed *exact* duplicate prompts. In the context of LLM training, why might dropping exact duplicates not be enough? What other types of duplicates exist that we should worry about?


###"Because you can write same  promte in  two different ways"

##**Part 2: Cleaning and Filtering Data**

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load the dataset
dataset = load_dataset('rotten_tomatoes', split='train')

# Convert to Pandas DataFrame
df_rotten = dataset.to_pandas()

print(f"Initial number of rows: {len(df_rotten)}")

# Filter out reviews shorter than 30 characters
df_rotten_filtered = df_rotten[df_rotten['text'].str.len() >= 30]

print(f"Number of rows after filtering short reviews: {len(df_rotten_filtered)}")

# Map numerical labels to 'negative' and 'positive'
label_mapping = {0: 'negative', 1: 'positive'}
df_rotten_filtered['label'] = df_rotten_filtered['label'].map(label_mapping)

# Display the first few rows of the processed DataFrame
display(df_rotten_filtered.head())

README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Initial number of rows: 8530
Number of rows after filtering short reviews: 8249


/tmp/ipykernel_34919/1452277422.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_rotten_filtered['label'] = df_rotten_filtered['label'].map(label_mapping)


,text,label
0,the rock is destined to be the 21st century's ...,positive
1,"the gorgeously elaborate continuation of "" the...",positive
2,effective but too-tepid biopic,positive
3,if you sometimes like to go to the movies to h...,positive
4,"emerges as something rare , an issue movie tha...",positive


**Group Discussion 3**

---

**Question:** Why did we decide to drop reviews that were shorter than 30 characters? Think about the "Garbage In, Garbage Out" principle from the reading material.

###Because Reviews shorter than 30 characters lack sufficient context or reasoning

##**Part 3: Formatting Data (Building Chat Templates)**

In [1]:
from datasets import load_dataset
import pandas as pd

# 1. Load the dataset
dataset_dolly = load_dataset("databricks/databricks-dolly-15k", split="train")
df_dolly = dataset_dolly.to_pandas()

# 2. Define the formatting function
def format_templates(row):
    instruction = row['instruction']
    response = row['response']

    # f-strings allow us to inject variables inside curly brackets {}. \n creates a line break.

    # Gemma Format
    gemma = f"<start_of_turn>user\n{instruction}<end_of_turn>\n<start_of_turn>model\n{response}<end_of_turn>"

    # Llama 3 Format
    llama3 = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{instruction}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{response}<|eot_id|>"

    # Return as a Pandas Series so it creates two separate columns
    return pd.Series([gemma, llama3])

# 3. Apply the function to the DataFrame
df_dolly[['gemma_formatted', 'llama3_formatted']] = df_dolly.apply(format_templates, axis=1)

# 4. Print to compare the differences!
print("--- GEMMA FORMAT ---")
print(df_dolly['gemma_formatted'].iloc[0])
print("\n--- LLAMA 3 FORMAT ---")
print(df_dolly['llama3_formatted'].iloc[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

--- GEMMA FORMAT ---
<start_of_turn>user
When did Virgin Australia start operating?<end_of_turn>
<start_of_turn>model
Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.<end_of_turn>

--- LLAMA 3 FORMAT ---
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

When did Virgin Australia start operating?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.<|eot_id|>


**Group Discussion 4**

---

**Question:** Look at the printout comparing the Gemma format to the Llama 3 format. They look completely alien to each other! What would happen if you successfully fine-tuned a Llama 3 model on the `llama3_formatted` column, but when you deployed it to your app, your app's code sent user messages using the `gemma_formatted` template?

###Because if you fine-tune a model on one chat template (Llama 3) but then use a different template (Gemma) in your application, the model will not understand the input format, leading to poor or nonsensical responses.